# S04 Lab — Northstar Technique Bake-Off

## The same 12 tickets, two techniques, measured

Northstar now has two working techniques for deciding a support ticket: a single structured call (S02) and a verified agent loop with an account-lookup tool (S01). This lab runs both on the exact same labeled tickets, measures accuracy, illustrative cost, and illustrative latency for each, and then builds a confidence-gated **tiered dispatch** between them.

## 4 parts, plus an FDE decision exercise and an optional live section:

PART 1: The shared ticket set

PART 2: Technique 1 — single structured call

PART 3: Technique 2 — verified agent loop

PART 4: Head to head

PART 5: Tiered dispatch

> [!NOTE]
> No API key, network access, or paid account is required for Parts 1–5. The optional live section (Part 7) is off by default (`RUN_LIVE = False`). All tickets and accounts below are synthetic.

In [ ]:
# %pip install pandas 'pydantic>=2,<3'

from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field

## Part 1: The shared ticket set

This is the same output contract S02 validated — reusing it, unchanged, is part of what makes this comparison fair.

In [ ]:
class TicketDecision(BaseModel):
    """The same decision contract validated in S02."""

    category: Literal["billing", "technical", "access", "general"]
    priority: Literal["low", "medium", "high"]
    needs_human: bool
    summary: str = Field(min_length=1, max_length=240)

Six tickets are **self-contained**: everything needed to decide them correctly is in the text. Six need an **account lookup**: the text alone is ambiguous or misleading, and the correct decision depends on an account fact the customer's wording doesn't reliably state.

In [ ]:
TICKETS = [
    {"ticket_id": "T-201", "text": "I was charged twice for this month's subscription — same amount, same day. Please refund the duplicate.", "account_id": None, "needs_lookup": False,
     "gold": {"category": "billing", "priority": "medium", "needs_human": False}},
    {"ticket_id": "T-202", "text": "The usage-report export button downloads a CSV with headers but no rows.", "account_id": None, "needs_lookup": False,
     "gold": {"category": "technical", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-203", "text": "Where can I find the changelog for last month's API release?", "account_id": None, "needs_lookup": False,
     "gold": {"category": "general", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-204", "text": "All five of our workspace admins got locked out right after today's SSO certificate rotation, and we have a board demo in an hour.", "account_id": None, "needs_lookup": False,
     "gold": {"category": "access", "priority": "high", "needs_human": True}},
    {"ticket_id": "T-205", "text": "The card on file expired. Can I update my payment method myself, or do I need you to do it?", "account_id": None, "needs_lookup": False,
     "gold": {"category": "billing", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-206", "text": "Webhook deliveries have been failing with a 522 error since this morning's deploy.", "account_id": None, "needs_lookup": False,
     "gold": {"category": "technical", "priority": "medium", "needs_human": False}},
    {"ticket_id": "T-207", "text": "Support told me my plan includes SSO, but the login page is asking me to set a password instead.", "account_id": "ACC-301", "needs_lookup": True,
     "gold": {"category": "general", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-208", "text": "You've been charging me for the Pro plan, but I'm sure I downgraded to Starter back in March.", "account_id": "ACC-302", "needs_lookup": True,
     "gold": {"category": "billing", "priority": "medium", "needs_human": True}},
    {"ticket_id": "T-209", "text": "This is the third time this quarter our integration token has expired early. What's going on?", "account_id": "ACC-303", "needs_lookup": True,
     "gold": {"category": "general", "priority": "low", "needs_human": False}},
    {"ticket_id": "T-210", "text": "Can you restore access for the workspace we offboarded last week? We still need the data for a compliance audit.", "account_id": "ACC-304", "needs_lookup": True,
     "gold": {"category": "access", "priority": "high", "needs_human": True}},
    {"ticket_id": "T-211", "text": "Our invoice lists 25 seats, but we've only ever provisioned 18 in the admin console.", "account_id": "ACC-305", "needs_lookup": True,
     "gold": {"category": "billing", "priority": "medium", "needs_human": True}},
    {"ticket_id": "T-212", "text": "We're being rate-limited well below the limit documented for our plan.", "account_id": "ACC-306", "needs_lookup": True,
     "gold": {"category": "technical", "priority": "high", "needs_human": True}},
]

ACCOUNTS = {
    "ACC-301": {"tier": "starter", "sso_included": False},
    "ACC-302": {"tier": "starter", "billed_rate": "pro", "downgrade_processed": True},
    "ACC-303": {"tier": "free", "documented_token_ttl_minutes": 15},
    "ACC-304": {"status": "offboarded", "compliance_hold": True},
    "ACC-305": {"provisioned_seats": 18, "invoiced_seats": 25},
    "ACC-306": {"tier": "growth", "documented_limit_per_min": 5000, "configured_limit_per_min": 500},
}

pd.DataFrame(TICKETS)[["ticket_id", "text", "needs_lookup"]]

## Part 2: Technique 1 — single structured call

A deterministic stand-in for a single structured call with **no tool access**: it decides self-contained tickets correctly, and has no way to resolve lookup-needed ones — so it guesses, trusting the customer's framing at face value. It also returns a `confident` flag from a simple keyword heuristic, which Part 5 uses for tiered dispatch.

In [ ]:
CONFIDENCE_TRIGGERS = ["plan includes", "downgraded", "third time", "offboarded", "provisioned", "documented for"]

SINGLE_CALL_COST = 1
SINGLE_CALL_SECONDS = 1.0

# Technique 1's guesses on the six lookup-needed tickets, made without account access.
_FACE_VALUE_GUESSES = {
    "T-207": {"category": "access", "priority": "high", "needs_human": True},
    "T-208": {"category": "billing", "priority": "low", "needs_human": False},
    "T-209": {"category": "technical", "priority": "high", "needs_human": True},
    "T-210": {"category": "access", "priority": "medium", "needs_human": False},
    "T-211": {"category": "billing", "priority": "low", "needs_human": False},
    "T-212": {"category": "technical", "priority": "medium", "needs_human": False},
}


def decide_single_call(ticket):
    """Mock Technique 1: a deterministic stand-in for a single structured call with no tool access."""
    text = ticket["text"].lower()
    confident = not any(trigger in text for trigger in CONFIDENCE_TRIGGERS)

    if not ticket["needs_lookup"]:
        decision = dict(ticket["gold"])  # self-contained: the text alone is enough
    else:
        decision = _FACE_VALUE_GUESSES[ticket["ticket_id"]]

    return {
        "ticket_id": ticket["ticket_id"],
        "technique": "single_call",
        "confident": confident,
        "cost": SINGLE_CALL_COST,
        "seconds": SINGLE_CALL_SECONDS,
        **decision,
    }

In [ ]:
def is_right(result, gold):
    """True when category, priority, and needs_human all match the reference label."""
    return (
        result["category"] == gold["category"]
        and result["priority"] == gold["priority"]
        and result["needs_human"] == gold["needs_human"]
    )


def score(results, tickets):
    """Attach a right/wrong column to each result, scored against the shared gold labels."""
    gold_by_id = {t["ticket_id"]: t["gold"] for t in tickets}
    rows = [{**r, "right": is_right(r, gold_by_id[r["ticket_id"]])} for r in results]
    return pd.DataFrame(rows)


def summarize(df, label):
    """Print and return accuracy, total illustrative cost, and total illustrative seconds for one technique's run."""
    right = int(df["right"].sum())
    cost = df["cost"].sum()
    seconds = df["seconds"].sum()
    print(f"{label}: {right} of {len(df)} right, {cost} cost-units total, {seconds:.1f} seconds total")
    return {"technique": label, "right": right, "total": len(df), "accuracy": right / len(df), "cost": cost, "seconds": seconds}

In [ ]:
technique1_results = [decide_single_call(t) for t in TICKETS]
technique1_df = score(technique1_results, TICKETS)
summary1 = summarize(technique1_df, "Technique 1 (single call)")
technique1_df[["ticket_id", "category", "priority", "needs_human", "confident", "right"]]

Technique 1 should get the six self-contained tickets right and the six lookup-needed tickets wrong — not because it's a bad model, but because it structurally cannot reach the account fact each of those six actually turns on.

## Part 3: Technique 2 — verified agent loop

A deterministic stand-in for S01's technique: it always calls `lookup_account` before deciding (the agent doesn't know in advance which tickets need it any more than Technique 1 does), then applies the correct business rule using the account record.

In [ ]:
AGENT_COST = 3
AGENT_SECONDS = 3.0


def lookup_account(account_id):
    """The mock lookup_account tool: returns the synthetic account record, or None."""
    return ACCOUNTS.get(account_id)


def decide_with_agent(ticket, accounts):
    """Mock Technique 2: always looks up the account, then reaches the correct decision."""
    account = lookup_account(ticket["account_id"]) if ticket["account_id"] else None
    decision = dict(ticket["gold"])  # with account access (where there's an account), the agent always gets it right
    return {
        "ticket_id": ticket["ticket_id"],
        "technique": "agent_loop",
        "looked_up": account is not None,
        "cost": AGENT_COST,
        "seconds": AGENT_SECONDS,
        **decision,
    }

In [ ]:
technique2_results = [decide_with_agent(t, ACCOUNTS) for t in TICKETS]
technique2_df = score(technique2_results, TICKETS)
summary2 = summarize(technique2_df, "Technique 2 (agent loop)")
technique2_df[["ticket_id", "category", "priority", "needs_human", "looked_up", "right"]]

## Part 4: Head to head

Both techniques ran on the exact same 12 tickets, the exact same contract, the exact same reference labels — only the technique changed.

In [ ]:
comparison = pd.DataFrame([summary1, summary2])
comparison

**Commercial implications:** Technique 2 wins on accuracy here by construction — six of the twelve tickets are designed to be unsolvable without the lookup it has and Technique 1 doesn't. That's the baseline the next part has to beat on cost without losing it on accuracy. A real recommendation still needs the volume math from the handout's §6 and the FDE decision exercise below — not just this percentage.

## Part 5: Tiered dispatch

Run the cheap technique first. If its confidence signal says the answer can be trusted, keep it. If not, escalate to the expensive technique — and return *that* answer instead.

In [ ]:
def tiered_dispatch(ticket, accounts):
    """Run Technique 1; escalate to Technique 2 only when Technique 1 isn't confident."""
    cheap = decide_single_call(ticket)
    if cheap["confident"]:
        return {**cheap, "technique": "tiered"}
    expensive = decide_with_agent(ticket, accounts)
    return {**expensive, "technique": "tiered"}


tiered_results = [tiered_dispatch(t, ACCOUNTS) for t in TICKETS]
tiered_df = score(tiered_results, TICKETS)
summary3 = summarize(tiered_df, "Tiered dispatch")
tiered_df[["ticket_id", "confident", "category", "priority", "needs_human", "right"]]

In [ ]:
final_comparison = pd.DataFrame([summary1, summary2, summary3])
final_comparison["savings_vs_always_agent"] = 1 - final_comparison["cost"] / summary2["cost"]
final_comparison

> [!CAUTION]
> Tiered dispatch should match Technique 2's accuracy here at roughly a third less cost — but that clean separation is a **teaching simplification**. The confidence heuristic was designed against these exact 12 tickets. The mini exercise below shows how it breaks.

**Mini exercise:** Add a 13th ticket deliberately designed to fool `CONFIDENCE_TRIGGERS` — either a self-contained ticket that happens to contain one of the trigger phrases (forcing an unnecessary, costly escalation), or a lookup-needed ticket that avoids all of them (silently staying on the cheap path and returning a wrong answer). Run it through `tiered_dispatch` and identify which of the two misroute directions (handout §7) you triggered.

In [ ]:
# Edit this ticket, then run the cell below it.
MINI_EXERCISE_TICKET = {
    "ticket_id": "T-213",
    "text": "Replace this with your own ticket text.",
    "account_id": None,
    "needs_lookup": False,
    "gold": {"category": "general", "priority": "low", "needs_human": False},
}

In [ ]:
tiered_dispatch(MINI_EXERCISE_TICKET, ACCOUNTS)

## Part 6: FDE decision exercise

Write the recommendation-plus-validation-plan for Northstar's VP of Support, who wants to move every ticket to the agent loop because it tested more accurate. Use the volume math pattern from the handout's §6, not just this lab's percentages.

Full exercise, rubric, and worked examples: [`docs/fde-exercises/model-technique-bakeoff-decision.md`](../../../docs/fde-exercises/model-technique-bakeoff-decision.md)

## Part 7: Optional live model mode (requires credentials and provider validation)

This section runs **Technique 1 only** — a real single structured call — against a real model, so you can compare a real latency/cost number to the mock numbers above. It does not run a live Technique 2: a real tool-using agent comparison needs a framework or hand-rolled harness wired to a live model, and S01's lab already has one built and tested — this lab doesn't duplicate it live.

> [!CAUTION]
> Only ever send synthetic or illustrative data to a live model, never real customer data. This path has not been executed as part of building this lab — review the provider's current structured-output support, data-retention terms, and pricing before running it live.

In [ ]:
# %pip install openai python-dotenv

import os

RUN_LIVE = False
LIVE_MODEL = "openai/gpt-4o-mini"  # replace with a verified, current OpenRouter model ID

if RUN_LIVE:
    from dotenv import load_dotenv
    from openai import OpenAI

    load_dotenv(override=True)
    openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
    if not openrouter_api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY in your environment before running the live section.")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

    def decide_single_call_live(ticket, model):
        """A real single structured call: Technique 1, against a real model."""
        response = client.chat.completions.parse(
            model=model,
            messages=[
                {"role": "system", "content": "You are a Northstar support triage assistant. Decide the ticket using only the text given — you have no other information."},
                {"role": "user", "content": ticket["text"]},
            ],
            response_format=TicketDecision,
        )
        return response.choices[0].message.parsed

    for ticket in TICKETS:
        print(ticket["ticket_id"], decide_single_call_live(ticket, LIVE_MODEL))
else:
    print("RUN_LIVE is False — skipping the live section. Set RUN_LIVE = True and OPENROUTER_API_KEY to run it.")

---

## About the mentor

**Sanjay Negi** brings 23+ years of industry experience and helps experienced software engineers and technology leaders transition into high-value AI roles, including **AI Engineer**, **Forward Deployed Engineer**, and **GenAI Architect**. He has trained more than 1,100 experienced engineers.

#SkillUpWithSanjay · [Website](https://www.sanjaynegi.in) · [LinkedIn](https://www.linkedin.com/in/sanjay-negi-ai/)